
- Gaurav BR (2023). *Hopkins Statistic Code*. Kaggle. https://www.kaggle.com/code/gauravbr/hopkins-statistic-code-clustering
- Chowksey, P. *Hopkins Statistic Clustering Tendency*. GitHub. https://github.com/prathmachowksey/Hopkins-Statistic-Clustering-Tendency



In [1]:
from sklearn.neighbors import NearestNeighbors
from random import sample
from numpy.random import uniform
import numpy as np
from math import isnan
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('movement.csv')

print('Shape:', df.shape)
df.head()

Shape: (469810, 14)


,licencePlate,car_type,vehicleTypeId,start_move_time,end_move_time,start_lat,start_lon,end_lat,end_lon,start_zip,end_zip,start_area_name,end_area_name,move_duration
0,bn32098,car,2,7/22/2025 9:13,7/22/2025 9:17,55.658398,12.514628,55.658348,12.515684,2500,2500,Valby,Valby,0 days 00:04:00
1,bn32098,car,2,7/22/2025 9:19,7/22/2025 9:23,55.658348,12.515684,55.659286,12.519309,2500,1805,Valby,Frederiksberg C,0 days 00:04:00
2,bn32098,car,2,7/22/2025 12:10,7/22/2025 14:24,55.659286,12.519309,55.677685,12.522237,1805,2000,Frederiksberg C,Frederiksberg C,0 days 02:14:01
3,bn32098,car,2,7/23/2025 5:44,7/23/2025 15:46,55.677685,12.522237,55.676945,12.520396,2000,2000,Frederiksberg C,Frederiksberg C,0 days 10:02:00
4,bn32098,car,2,7/23/2025 17:06,7/23/2025 17:20,55.676945,12.520396,55.655346,12.537441,2000,2450,Frederiksberg C,Kobenhavn SV,0 days 00:14:00


In [3]:
cols = ['start_lat', 'start_lon', 'end_lat', 'end_lon']

data = df[cols].dropna().reset_index(drop=True)

print('Rows after dropping missing values:', len(data))
data.head()

Rows after dropping missing values: 469810


,start_lat,start_lon,end_lat,end_lon
0,55.658398,12.514628,55.658348,12.515684
1,55.658348,12.515684,55.659286,12.519309
2,55.659286,12.519309,55.677685,12.522237
3,55.677685,12.522237,55.676945,12.520396
4,55.676945,12.520396,55.655346,12.537441


In [4]:
scaler = StandardScaler()
data_scaled = pd.DataFrame(
    scaler.fit_transform(data),
    columns=cols
)

print('Scaled data shape:', data_scaled.shape)
print('\nMean of each column (should be ~0):')
print(data_scaled.mean().round(4))
print('\nStd of each column (should be ~1):')
print(data_scaled.std().round(4))

Scaled data shape: (469810, 4)

Mean of each column (should be ~0):
start_lat   -0.0
start_lon   -0.0
end_lat      0.0
end_lon      0.0
dtype: float64

Std of each column (should be ~1):
start_lat    1.0
start_lon    1.0
end_lat      1.0
end_lon      1.0
dtype: float64


In [9]:
def hopkins(X):
   
    d = X.shape[1]          
    n = len(X)              
    m = int(0.1 * n)        
    
    
    nbrs = NearestNeighbors(n_neighbors=1).fit(X.values)
    
    rand_X = sample(range(0, n, 1), m)
    
    ujd = []  
    wjd = []  
    
    for j in range(0, m):
        u_dist, _ = nbrs.kneighbors(
            uniform(np.amin(X, axis=0), np.amax(X, axis=0), d).reshape(1, -1),
            2, return_distance=True
        )
        ujd.append(u_dist[0][1])
        
        
        w_dist, _ = nbrs.kneighbors(
            X.iloc[rand_X[j]].values.reshape(1, -1),
            2, return_distance=True
        )
        wjd.append(w_dist[0][1])
    
   
    H = sum(ujd) / (sum(ujd) + sum(wjd))
    
   
    if isnan(H):
        print('Warning: NaN detected. Check for duplicate coordinates.')
        H = 0
    
    return H


In [ ]:
np.random.seed(42)

H = hopkins(data_scaled)

print(f'Hopkins Statistic H = {H:.4f}')
print()



KeyboardInterrupt: 